In [ ]:
pip install datasets scikit-learn transformers[torch] 

In [12]:
pip install accelerate>=0.26.0

Note: you may need to restart the kernel to use updated packages.


In [1]:
import torch
from datasets import load_dataset
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import accuracy_score

In [3]:
# Load the IMDb Dataset
print("Loading IMDb dataset...")
imdb = load_dataset("imdb")

# Preprocessing
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize_function(examples):
    # Truncation=True ensures sequences don't exceed BERT's max length (512)
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("Tokenizing data...")
tokenized_imdb = imdb.map(tokenize_function, batched=True)

# A small subset for has been used for demonstration purposes (to make training fast)
small_train_dataset = tokenized_imdb["train"].shuffle(seed=42).select(range(500))
small_test_dataset = tokenized_imdb["test"].shuffle(seed=42).select(range(100))

# Model Initialization
# num_labels=2 for "Positive" (1) and "Negative" (0)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Loading IMDb dataset...
Tokenizing data...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [4]:
# Predicting the sentiment score before fine-tuning 
def predict_sentiment(text, model, tokenizer):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128, padding=True).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_class_id = logits.argmax().item()
    return "Positive" if predicted_class_id == 1 else "Negative"

print("\n" + "="*40)
print("SENTIMENT ANALYSIS: BEFORE TRAINING")
print("="*40)
example_1 = "This movie was absolutely wonderful, I loved every moment."
example_2 = "This was a complete disaster. Waste of time."

print(f"Input: {example_1}")
print(f"Prediction: {predict_sentiment(example_1, model, tokenizer)} (Likely Random)")
print("-" * 20)
print(f"Input: {example_2}")
print(f"Prediction: {predict_sentiment(example_2, model, tokenizer)} (Likely Random)")


# Fine-Tuning (Training)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    output_dir="./imdb_results",
    num_train_epochs=3,              
    per_device_train_batch_size=8,
    eval_strategy="epoch",  # Changed from evaluation_strategy to eval_strategy
    logging_steps=50,
    learning_rate=2e-5,
    use_cpu=not torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    compute_metrics=compute_metrics,
)

print("\nStarting Training (Fine-tuning)...")
trainer.train()

# Predicting the sentiment score after fine-tuning 
print("\n" + "="*40)
print("SENTIMENT ANALYSIS: AFTER TRAINING")
print("="*40)

print(f"Input: {example_1}")
print(f"Prediction: {predict_sentiment(example_1, model, tokenizer)}")
print("-" * 20)
print(f"Input: {example_2}")
print(f"Prediction: {predict_sentiment(example_2, model, tokenizer)}")


SENTIMENT ANALYSIS: BEFORE TRAINING
Input: This movie was absolutely wonderful, I loved every moment.
Prediction: Positive (Likely Random)
--------------------
Input: This was a complete disaster. Waste of time.
Prediction: Positive (Likely Random)

Starting Training (Fine-tuning)...


Epoch,Training Loss,Validation Loss,Accuracy
1,0.664000,0.611066,0.650000
2,0.511700,0.457855,0.810000
3,0.327500,0.388462,0.840000



SENTIMENT ANALYSIS: AFTER TRAINING
Input: This movie was absolutely wonderful, I loved every moment.
Prediction: Positive
--------------------
Input: This was a complete disaster. Waste of time.
Prediction: Positive


In [ ]:
# Load the SNLI Dataset
print("\nLoading SNLI dataset...")
snli = load_dataset("snli")

# Filter out entries with label -1 (where annotators disagreed)
snli = snli.filter(lambda x: x['label'] != -1)

# Preprocessing
# BERT requires pairs to be separated by [SEP]. The tokenizer handles this automatically when two arguments are passed. 
def preprocess_nli(examples):
    return tokenizer(
        examples['premise'], 
        examples['hypothesis'], 
        padding="max_length", 
        truncation=True, 
        max_length=128
    )

print("Tokenizing SNLI data (Pairs)...")
tokenized_snli = snli.map(preprocess_nli, batched=True)

# Subset for demo
small_nli_train = tokenized_snli["train"].shuffle(seed=42).select(range(500))
small_nli_test = tokenized_snli["test"].shuffle(seed=42).select(range(100))

# Model Initialization
# Labels: 0=Entailment, 1=Neutral, 2=Contradiction (Standard HuggingFace mapping)
# Note that the reading material provided uses a different mapping from the standard HF.
id2label = {0: "ENTJAILMENT", 1: "NEUTRAL", 2: "CONTRADICTION"}
model_nli = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=3)
model_nli.to(device)

# Predicting the inference score before fine-tuning 
def predict_nli(premise, hypothesis, model, tokenizer):
    # Pass both sequences to the  tokenizer
    inputs = tokenizer(
        premise, 
        hypothesis, 
        return_tensors="pt", 
        truncation=True, 
        max_length=128, 
        padding=True
    ).to(device)
    
    with torch.no_grad():
        logits = model(**inputs).logits
    predicted_id = logits.argmax().item()
    return id2label[predicted_id]

print("\n" + "="*40)
print("NLI: BEFORE TRAINING")
print("="*40)
premise = "A soccer player is running across the field."
hypothesis_entail = "A person is moving."
hypothesis_contra = "A person is sitting down."

print(f"Premise: {premise}")
print(f"Hypothesis: {hypothesis_entail}")
print(f"Prediction: {predict_nli(premise, hypothesis_entail, model_nli, tokenizer)} (Likely Random)")
print("-" * 20)
print(f"Premise: {premise}")
print(f"Hypothesis: {hypothesis_contra}")
print(f"Prediction: {predict_nli(premise, hypothesis_contra, model_nli, tokenizer)} (Likely Random)")

In [6]:
# Fine-Tuning
training_args_nli = TrainingArguments(
    output_dir="./snli_results",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    logging_steps=50,
    learning_rate=2e-5,
    use_cpu=not torch.cuda.is_available()
)

trainer_nli = Trainer(
    model=model_nli,
    args=training_args_nli,
    train_dataset=small_nli_train,
    eval_dataset=small_nli_test,
    compute_metrics=compute_metrics,
)

print("\nStarting NLI Training (Fine-tuning)...")
trainer_nli.train()

## Predicting the sentiment score after fine-tuning 
print("\n" + "="*40)
print("NLI: AFTER TRAINING")
print("="*40)

print(f"Premise: {premise}")
print(f"Hypothesis: {hypothesis_entail}")
print(f"Prediction: {predict_nli(premise, hypothesis_entail, model_nli, tokenizer)}")
print("-" * 20)
print(f"Premise: {premise}")
print(f"Hypothesis: {hypothesis_contra}")
print(f"Prediction: {predict_nli(premise, hypothesis_contra, model_nli, tokenizer)}")


Starting NLI Training (Fine-tuning)...


Epoch,Training Loss,Validation Loss,Accuracy
1,1.104300,1.037469,0.510000
2,0.962800,0.931176,0.590000
3,0.794600,0.896231,0.550000



NLI: AFTER TRAINING
Premise: A soccer player is running across the field.
Hypothesis: A person is moving.
Prediction: CONTRADICTION
--------------------
Premise: A soccer player is running across the field.
Hypothesis: A person is sitting down.
Prediction: CONTRADICTION
